# Bronze to Silver
Azure Synapse Analytics: process all registered Bronze Delta tables sequentially into Silver Delta tables. The first run reads existing rows; subsequent runs process new appends using a separate persistent checkpoint per table.

Configure the databases, storage paths, and optional per-table casts in Cell 2. Replace the example table and column names with real names, or use an empty dictionary to preserve source types. Unmapped tables preserve their Bronze schema, including ingestion metadata. Invalid casts fail the table under ANSI mode.

Use new or empty Silver targets on the first run. Keep checkpoints between runs and do not overlap runs. Bronze must remain append-only: this does not handle updates, deletes, or business-key deduplication. Schema changes require deliberate handling; changing casts does not recast historical Silver rows. A failed table stops the notebook; rerun with the same checkpoints after correcting the failure.

In [ ]:
from pyspark.sql import functions as F

bronze_database = 'adventureworks_bronze'
silver_database = 'adventureworks_silver'
silver_root = 'abfss://<silver-container>@datalakejdobbins.dfs.core.windows.net/azsql/saleslt'
checkpoint_root = 'abfss://<checkpoint-container>@datalakejdobbins.dfs.core.windows.net/bronze_to_silver/azsql/saleslt'

column_types = {
    'example_table': {
        'example_id': 'bigint',
        'example_amount': 'decimal(18,2)',
        'example_date': 'date',
        'example_timestamp': 'timestamp',
    },
}

In [ ]:
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark.conf.set('spark.sql.ansi.enabled', 'true')
spark.sql(f'CREATE DATABASE IF NOT EXISTS `{silver_database}`')

for table in spark.catalog.listTables(bronze_database):
    if table.isTemporary or table.tableType == 'VIEW':
        continue

    table_name = table.name
    escaped_table_name = table_name.replace('`', '``')
    source_table = f'`{bronze_database}`.`{escaped_table_name}`'
    target_table = f'`{silver_database}`.`{escaped_table_name}`'

    if spark.sql(f'DESCRIBE DETAIL {source_table}').first()['format'] != 'delta':
        print(f'Skipping non-Delta table: {source_table}', flush=True)
        continue

    print(f'Processing table: {source_table} -> {target_table}', flush=True)

    silver = spark.readStream.format('delta').table(source_table)

    for column_name, target_type in column_types.get(table_name, {}).items():
        escaped_column_name = column_name.replace('`', '``')
        silver = silver.withColumn(
            column_name, F.col(f'`{escaped_column_name}`').cast(target_type)
        )

    query = (
        silver.writeStream.format('delta').outputMode('append')
        .option('path', f'{silver_root}/{table_name}')
        .option('checkpointLocation', f'{checkpoint_root}/{table_name}')
        .trigger(availableNow=True)
        .toTable(target_table)
    )
    query.awaitTermination()
    print(f'Completed table: {target_table}', flush=True)